# Wave equation

$$u_{tt} - c^2\,u_{xx} = 0,\qquad x\in[0,1],\ t\in(0,1)$$

with Dirichlet walls $u(0,t)=u(1,t)=0$ and a standing-wave start $u(x,0)=\sin(\pi x)+\tfrac12\sin(\beta\pi x)$, $u_t(x,0)=0$.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import jax
import jax.numpy as jnp

import pinn
from pinn import operators as op, sampling

%matplotlib inline
jax.config.update("jax_enable_x64", True)   # double precision

In [ ]:
X, T = 0, 1


class WaveProblem(pinn.Problem):
    """u_tt - c^2 u_xx = 0 with Dirichlet walls and a standing-wave start."""

    x_min, x_max, t_max = 0.0, 1.0, 1.0
    c, beta = 2.0, 4.0
    problem_name = "Wave"
    ref_path = pinn.reference_path("wave")

    def __init__(self, *, n_pde, n_ic, n_bc):
        self.n_pde, self.n_ic, self.n_bc = n_pde, n_ic, n_bc

    def residual_fns(self):
        return {"pde": self.pde_residual, "ic": self.ic_residual,
                "ic_t": self.ic_t_residual, "bc": self.bc_residual}

    def pde_residual(self, model, coords):
        u = lambda c: model(c)[0]
        r = op.grad2(u, coords, T) - self.c**2 * op.grad2(u, coords, X)
        return jnp.array([r])

    def ic_residual(self, model, coords):
        return jnp.array([model(coords[:2])[0] - coords[2]])

    def ic_t_residual(self, model, coords):
        u = lambda c: model(c)[0]
        return jnp.array([op.grad(u, coords[:2], T) - coords[2]])

    def bc_residual(self, model, coords):
        return jnp.array([model(coords)[0]])

    def samplers(self):
        line = sampling.initial_line(self.x_min, self.x_max)
        return {"pde": sampling.interior(self.x_min, self.x_max),
                "ic": line, "ic_t": line,
                "bc": sampling.dirichlet_walls(self.x_min, self.x_max)}

    def analytical_ic(self, x):
        return jnp.sin(jnp.pi * x) + 0.5 * jnp.sin(self.beta * jnp.pi * x)

    def analytical_ic_t(self, x):
        return jnp.zeros_like(x)

In [ ]:
cfg = pinn.RunConfig(
    dt=0.5,
    network=lambda key: pinn.SIREN(
        key, WaveProblem, hidden_dims=(60, 60, 60, 60),
        periodic_bc=False, n_inputs=2, n_outputs=1,
    ),
    n_pde=2**15, n_ic=2**14, n_bc=2**14,
    residual_sketch=4000, parameter_sketch=4000,
    batch_size=2**14, probe_batch_size=2**10,
    pde_weight=1e-6, ic_weight=1.0, ic_t_weight=1e-3, bc_weight=1.0,
)

In [ ]:
pinn.precompile64(WaveProblem, cfg)

In [ ]:
results = pinn.train64(WaveProblem, cfg)

## Results

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ref = pinn.load_reference("wave")
x0, x1 = results["plot_x0"], results["plot_x1"]
lx, ly = results["plot_axes"]
extent = [x0[0], x0[-1], x1[0], x1[-1]]

for ch in ref.channels:
    pred = np.array(results["u_pred_plot"][ch])
    exact = np.array(ref.plot_grids[ch])
    err = np.abs(pred - exact)
    rel = np.linalg.norm(pred - exact) / np.linalg.norm(exact)

    fig, axs = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
    for ax, data, title, cmap in zip(
        axs, [pred, exact, err],
        [f"PINN  ${ch}$", f"reference  ${ch}$", f"abs error  (rel $\\ell_2$={rel:.2e})"],
        ["RdBu_r", "RdBu_r", "magma"],
    ):
        im = ax.imshow(data.T, origin="lower", aspect="auto", extent=extent, cmap=cmap)
        ax.set(xlabel=f"${lx}$", ylabel=f"${ly}$", title=title)
        fig.colorbar(im, ax=ax)
    plt.show()